In [ ]:
import sys
import os
from typing import NamedTuple
import math
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax
import jax.numpy as jnp
import optax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_worm import artificializer
from utils_worm import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import bom_rs as bom
from models import hgm_rs as hgm
from models import hgm as hgm_gen


# jax.config.update("jax_enable_x64", True)

In [ ]:
import importlib
importlib.reload(artificializer)
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(bom)
importlib.reload(hgm)
importlib.reload(hgm_gen)

In [ ]:
manual_hyperparameters = {
	# 'max_epochs': 3000,
	'max_epochs': 3000,
	'max_patience': 3000 / 4,
	'learning_rate': 1E-0,
	'dimensions': 2,
	'ccadiag_lambda': 0.6,
	'ccaunit_lambda': 1.0 - 0.6,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'vb_reg_lambda': 0.0,
	'wm_reg_lambda': 0.0,
}

# Functions

In [ ]:
class Lambdas(NamedTuple):
	loss_lambdas: tuple[float, float]
	reg_lambdas: tuple[float, float]
	cca_lambdas: tuple[float, float]

In [ ]:
def train_model(model, splits, N, B, S, hyperparams=None):
	#region hyperparamters ===============================================================
	seeds = jax.random.PRNGKey(0), jax.random.PRNGKey(1), jax.random.PRNGKey(2), jax.random.PRNGKey(3), jax.random.PRNGKey(4)

	global manual_hyperparameters
	hyperparameters = manual_hyperparameters.copy()
	if hyperparams is not None:
		for key in hyperparams:
			hyperparameters[key] = hyperparams[key]

	max_epochs = hyperparameters['max_epochs']
	max_patience = hyperparameters['max_patience']

	learning_rate = hyperparameters['learning_rate']

	n_dimensions = hyperparameters['dimensions']

	lambdas = Lambdas(
		loss_lambdas=(hyperparameters['loss_man_lambda'], hyperparameters['loss_res_lambda']),
		reg_lambdas=(hyperparameters['vb_reg_lambda'], hyperparameters['wm_reg_lambda']),
		cca_lambdas=(hyperparameters['ccadiag_lambda'], hyperparameters['ccaunit_lambda'])
	)
	#endregion ===========================================================================

	n_neurons = N.shape[1]
	n_states = S.shape[1]
	n_behaviors = B.shape[1]

	# training loop ==============================
	params_per_split = list()
	(losses_train_per_split,
	 man_losses_train_per_split,
	 res_losses_train_per_split,
	 mpe_losses_train_per_split) = list(), list(), list(), list()
	(losses_per_split,
	 man_losses_per_split,
	 res_losses_per_split,
	 mpe_losses_per_split) = list(), list(), list(), list()
	for s_id, (train, eval) in enumerate(splits):
		#region splitting and initializing (this also moves the data to the GPU) ==========================
		N_train, N_eval = jnp.array(N[train]), jnp.array(N[eval])
		S_train, S_eval = jnp.array(S[train]), jnp.array(S[eval])
		B_train, B_eval = jnp.array(B[train]), jnp.array(B[eval])

		params = model.init_params(N_train, B_train, S_train,
								   seeds,
								   (n_neurons, n_dimensions),
								   (n_states, n_dimensions),
								   (n_behaviors, n_neurons),
								   (n_behaviors, n_behaviors),
								   * ((n_dimensions, n_behaviors),) if model.get_type() else (),
								  )

		optimizer = optax.adam(learning_rate=optax.cosine_decay_schedule(init_value=learning_rate, decay_steps=max_epochs, alpha=1E-4))
		# optimizer = optax.adam(learning_rate=learning_rate)
		optimizer = optax.masked(optimizer, {'prep': False, 'model': True})
		opt_state = optimizer.init(params)
		#endregion ========================================================================================

		(losses_train,
		 man_losses_train,
		 res_losses_train,
		 mpe_losses_train) = [], [], [], []
		(losses,
		 man_losses,
		 res_losses,
		 mpe_losses) = [], [], [], []

		best_params = params
		(best_loss_train,
		 best_man_loss_train,
		 best_res_loss_train,
		 best_mpe_loss_train) = sys.float_info.max, sys.float_info.max, sys.float_info.max, sys.float_info.max
		(best_loss,
		 best_man_loss,
		 best_res_loss,
		 best_mpe_loss) = sys.float_info.max, sys.float_info.max, sys.float_info.max, sys.float_info.max

		# lr_step = 0
		patience = 0
		for epoch in range(max_epochs):

			(loss,
			 man_loss,
			 res_loss,
			 mpe_loss) = model.loss_fn_flattened(params, N_eval, B_eval, S_eval, lambdas)

			(opt_state, params, loss_types_train) = model.update(optimizer.update, opt_state, params, N_train, B_train, S_train, lambdas)

			(loss_train,
			 man_loss_train,
			 res_loss_train,
			 mpe_loss_train) = loss_types_train

			#region early stopping ===========================================================================
			if epoch >= max_patience:  # give it an initial time
				if loss < best_loss:
					patience = 0

					best_params = params
					(best_loss_train,
					 best_man_loss_train,
					 best_res_loss_train,
					 best_mpe_loss_train) = loss_train, man_loss_train, res_loss_train, mpe_loss_train
					(best_loss,
					 best_man_loss,
					 best_res_loss,
					 best_mpe_loss) = loss, man_loss, res_loss, mpe_loss
				else:
					patience += 1
					if patience > max_patience:
						params = best_params

						losses_train = losses_train + [best_loss_train for _ in range(max_epochs - epoch)]
						man_losses_train = man_losses_train + [best_man_loss_train for _ in range(max_epochs - epoch)]
						res_losses_train = res_losses_train + [best_res_loss_train for _ in range(max_epochs - epoch)]
						mpe_losses_train = mpe_losses_train + [best_mpe_loss_train for _ in range(max_epochs - epoch)]

						losses = losses + [best_loss for _ in range(max_epochs - epoch)]
						man_losses = man_losses + [best_man_loss for _ in range(max_epochs - epoch)]
						res_losses = res_losses + [best_res_loss for _ in range(max_epochs - epoch)]
						mpe_losses = mpe_losses + [best_mpe_loss for _ in range(max_epochs - epoch)]

						print(f'Stopped after {epoch} epochs: ', end='')
						break

						# lr_step += 1
						# optimizer = optax.adam(learning_rate / (10 ** lr_step))
						# optimizer = optax.masked(optimizer, {'prep': False, 'model': True})
						# patience = 0
						# print(f'\nReduced learning rate from {learning_rate / (10 ** (lr_step - 1))} to {learning_rate / (10 ** lr_step)} after epoch {epoch} in split {s_id}', end='')
			#endregion =======================================================================================

			losses_train.append(loss_train)
			man_losses_train.append(man_loss_train)
			res_losses_train.append(res_loss_train)
			mpe_losses_train.append(mpe_loss_train)

			losses.append(loss)
			man_losses.append(man_loss)
			res_losses.append(res_loss)
			mpe_losses.append(mpe_loss)

		params_per_split.append(params)

		losses_train_per_split.append(jax.device_get(losses_train))
		man_losses_train_per_split.append(jax.device_get(man_losses_train))
		res_losses_train_per_split.append(jax.device_get(res_losses_train))
		mpe_losses_train_per_split.append(jax.device_get(mpe_losses_train))

		losses_per_split.append(jax.device_get(losses))
		man_losses_per_split.append(jax.device_get(man_losses))
		res_losses_per_split.append(jax.device_get(res_losses))
		mpe_losses_per_split.append(jax.device_get(mpe_losses))

		print(f'Split {s_id} completed. ', end='')

	# ============================================

	# return (this also moves the data to the CPU)
	return (params_per_split,
			np.array(losses_train_per_split), np.array(man_losses_train_per_split), np.array(res_losses_train_per_split), np.array(mpe_losses_train_per_split),
			np.array(losses_per_split), np.array(man_losses_per_split), np.array(res_losses_per_split), np.array(mpe_losses_per_split))

In [ ]:
def train_models(model, filenames, discrete_states, eigenworms, traces, hyperparams=None, save_name=None):
	models_params = list()
	models_losses, models_man_losses, models_res_losses, models_mpe_losses = list(), list(), list(), list()
	models_losses_train, models_man_losses_train, models_res_losses_train, models_mpe_losses_train = list(), list(), list(), list()

	for i in range(len(filenames)):
		print(f'\n{i+1}/{len(filenames)}: {filenames[i]}... ', end='')

		splits = list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces[i].shape[0]), discrete_states[i][['REV']].values))

		(params_per_split,
		 losses_train_per_split, man_losses_train_per_split, res_losses_train_per_split, mpe_losses_train_per_split,
		 losses_per_split, man_losses_per_split, res_losses_per_split, mpe_losses_per_split
		 ) = train_model(model, splits,
						 traces[i].values,
						 eigenworms[i].values,
						 discrete_states[i].values,
						 hyperparams)

		models_params.append(params_per_split)

		models_losses_train.append(losses_train_per_split)
		models_man_losses_train.append(man_losses_train_per_split)
		models_res_losses_train.append(res_losses_train_per_split)
		models_mpe_losses_train.append(mpe_losses_train_per_split)

		models_losses.append(losses_per_split)
		models_man_losses.append(man_losses_per_split)
		models_res_losses.append(res_losses_per_split)
		models_mpe_losses.append(mpe_losses_per_split)

	if save_name is not None:
		train_utils.save_params(filenames, models_params, save_name)
		train_utils.save_losses(filenames, [
			np.array([models_losses_train[i], models_man_losses_train[i], models_res_losses_train[i], models_mpe_losses_train[i],
					  models_losses[i], models_man_losses[i], models_res_losses[i], models_mpe_losses[i]]) for i in range(len(filenames))
		], save_name)

	return (models_params,
			models_losses_train, models_man_losses_train, models_res_losses_train, models_mpe_losses_train,
			models_losses, models_man_losses, models_res_losses, models_mpe_losses)

# read & prep

In [ ]:
ll = f'll{int(manual_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	# 'vb_reg_lambda': manual_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['bom_rs']['means']['vb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	# 'vb_reg_lambda': manual_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['vb_reg_lambda']['geometric'],
	# 'wm_reg_lambda': manual_hyperparameters['wm_reg_lambda'],
	'wm_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['wm_reg_lambda']['geometric'],
}

In [ ]:
artificial = True
if artificial:
	dims_generation = 2
	weights = (0.1, -0.25)


gcamp = True
file_ids_gcamp = list(range(0, 33+1))
# file_ids_gcamp = [0,   # minrevs
# 				  1,   # minswitch
# 				  7,   # maxswitch
# 				  11,  # median
# 				  12,  # maxrevs
# 				  17,  # mean
# 				  19,  # paper
# 				  21]  # clean
# file_ids_gcamp = [19]
plot_id_gcamp = 19


gfp = True
if gfp:
	file_ids_gfp = list(range(0, 7+1))
	# file_ids_gfp = [2, 4, 6]
	# file_ids_gfp = [2]
	plot_id_gfp = 2

In [ ]:
# read

path = os.path.join(read_utils.get_data_path(), 'exported_data_gcamp')
(file_ids_gcamp, filenames_gcamp,
 traces_gcamp,
 continuous_behaviors_gcamp,
 discrete_states_gcamp, eigenworms_gcamp,
 kymographs_gcamp) = read_utils.get_datas(path, file_ids_gcamp)

# clear nan values
for i in range(len(file_ids_gcamp)):
	mask = eigenworms_gcamp[i].notna().all(axis=1)

	traces_gcamp[i] = traces_gcamp[i][mask]
	#continuous_behaviors_gcamp[i] = continuous_behaviors_gcamp[i][mask]
	eigenworms_gcamp[i] = eigenworms_gcamp[i][mask]
	discrete_states_gcamp[i] = discrete_states_gcamp[i][mask]

	#kymographs_gcamp[i] = kymographs_gcamp[i][mask]
	#filenames_gcamp[i] = filenames_gcamp[i][mask]

# naive ID unknown neurons
for i in range(len(file_ids_gcamp)):
	traces_gcamp[i].columns = pd.Index([
		f'{file_ids_gcamp[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces_gcamp[i].columns, traces_gcamp[i].columns == ''))
	])


if gfp:
	path = os.path.join(read_utils.get_data_path(), 'exported_data_gfp')
	(file_ids_gfp, filenames_gfp,
	 traces_gfp,
	 continuous_behaviors_gfp,
	 discrete_states_gfp, eigenworms_gfp,
	 kymographs_gfp) = read_utils.get_datas(path, file_ids_gfp)

	# clear nan values
	for i in range(len(file_ids_gfp)):
		mask = eigenworms_gfp[i].notna().all(axis=1)

		traces_gfp[i] = traces_gfp[i][mask]
		#continuous_behaviors_gfp[i] = continuous_behaviors_gfp[i][mask]
		eigenworms_gfp[i] = eigenworms_gfp[i][mask]
		discrete_states_gfp[i] = discrete_states_gfp[i][mask]

		#kymographs_gfp[i] = kymographs_gfp[i][mask]
		#filenames_gfp[i] = filenames_gfp[i][mask]

	# naive ID unknown neurons
	for i in range(len(file_ids_gfp)):
		traces_gfp[i].columns = pd.Index([
			f'{file_ids_gfp[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces_gfp[i].columns, traces_gfp[i].columns == ''))
		])

In [ ]:
selection_n = ['VB02', 'DB01', 'DD01', 'AVBR', 'AVBL',	# paper
			   'RMDDR', 'SMDDR', 'SMDDL',				# C
			   'AVAR', 'AVAL',							# M
			   'ALA', 'AWBR']							# O
selection_bhl = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

plot_idx_gcamp = file_ids_gcamp.index(plot_id_gcamp)
selection_n_gcamp = [label for label in selection_n if label in traces_gcamp[plot_idx_gcamp].columns]

if gfp:
	plot_idx_gfp = file_ids_gfp.index(plot_id_gfp)
	selection_n_gfp = [label for label in selection_n if label in traces_gfp[plot_idx_gfp].columns]

# discrete_states_gcamp = [ds[selection_bhl] for ds in discrete_states_gcamp]

In [ ]:
# generate

if artificial:
	plot_id_art = plot_id_gcamp

	file_ids_art = list()
	filenames_art = list()

	discrete_states_art = list()
	eigenworms_art = list()

	modes_art = list()
	traces_reconstructed_art = list()
	traces_residual_art = list()
	traces_art = list()

	for i in range(len(file_ids_gcamp)):
		(filename_art,
		 neural_modes_art_,
		 traces_reconstructed_art_,
		 traces_residual_art_,
		 traces_art_,
		 params_art_,
		 params_cca_art_) = artificializer.artificialize(weights, hgm_gen,
														 filenames_gcamp[i],
														 traces_gcamp[i].columns,
														 eigenworms_gcamp[i],
														 [ds[selection_bhl] for ds in discrete_states_gcamp][i],
														 n_dimensions=dims_generation)
		filenames_art.append(filename_art)

		modes_art.append(neural_modes_art_)
		traces_reconstructed_art.append(traces_reconstructed_art_)
		traces_residual_art.append(traces_residual_art_)
		traces_art.append(traces_art_)

		file_ids_art.append(file_ids_gcamp[i])

		discrete_states_art.append(discrete_states_gcamp[i].copy())
		eigenworms_art.append(eigenworms_gcamp[i].copy())

	plot_idx_art = file_ids_art.index(plot_id_art)

# ARTIFICIAL

In [ ]:
if artificial:
	plot_utils.plot_traces(traces_residual_art[plot_idx_art][selection_n_gcamp],
						   discrete_states_art[plot_idx_art][['REV']],
						   label='reversal',
						   title=f'[{filenames_art[plot_idx_art]}] residuals traces artificial', ylimits=None)

In [ ]:
if artificial:
	plot_utils.plot_traces(traces_art[plot_idx_art][selection_n_gcamp],
						   discrete_states_art[plot_idx_art][['REV']],
						   label='reversal',
						   title=f'[{filenames_art[plot_idx_art]}] residuals artificial', ylimits=None)

### BOM

In [ ]:
if artificial:
	(boms_params_art,
	 boms_losses_train_art, boms_man_losses_train_art, boms_res_losses_train_art, boms_mpe_losses_train_art,
	 boms_losses_test_art, boms_man_losses_test_art, boms_res_losses_test_art, boms_mpe_losses_test_art
	 ) = train_models(bom,
					  filenames_art,
					  discrete_states_art,
					  eigenworms_art,
					  traces_art,
					  hyperparams=bom_reg_lambdas,
					  save_name=os.path.join('worm-2025-fieseler', 'new', ll, f'ss_bom_art_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}'))

### HGM

In [ ]:
if artificial:
	(hgms_params_art,
	 hgms_losses_train_art, hgms_man_losses_train_art, hgms_res_losses_train_art, hgms_mpe_losses_train_art,
	 hgms_losses_test_art, hgms_man_losses_test_art, hgms_res_losses_test_art, hgms_mpe_losses_test_art,
	 ) = train_models(hgm,
					  filenames_art,
					  discrete_states_art,
					  eigenworms_art,
					  traces_art,
					  hyperparams=hgm_reg_lambdas,
					  save_name=os.path.join('worm-2025-fieseler', 'new', ll, f'ss_hgm_art_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}'))

## losses

In [ ]:
if artificial:
	plot_utils.plot_losses_splits(filenames_art,
								  hgms_losses_train_art, boms_losses_train_art,
								  title='art hgm vs bom: train',
								  ylimits=None)

In [ ]:
if artificial:
	plot_utils.plot_losses_splits(filenames_art,
								  hgms_losses_test_art, boms_losses_test_art,
								  title='art hgm vs bom: test',
								  ylimits=None)

In [ ]:
if artificial:
	plot_utils.plot_losses_splits(filenames_art,
								  hgms_mpe_losses_train_art, boms_mpe_losses_train_art,
								  title='art hgm vs bom: mpe train',
								  ylimits=None)

In [ ]:
if artificial:
	plot_utils.plot_losses_splits(filenames_art,
								  hgms_mpe_losses_test_art, boms_mpe_losses_test_art,
								  title='art hgm vs bom: mpe test',
								  ylimits=None)

# GCAMP

In [ ]:
if gcamp:
	plot_utils.plot_traces(traces_gcamp[plot_idx_gcamp][selection_n_gcamp],
						   discrete_states_gcamp[plot_idx_gcamp][['REV']],
						   label='reversal',
						   title=f'[{filenames_gcamp[plot_idx_gcamp]}] gcamp traces', ylimits=None)

### BOM

In [ ]:
if gcamp:
	(boms_params_gcamp,
	 boms_losses_train_gcamp, boms_man_losses_train_gcamp, boms_res_losses_train_gcamp, boms_mpe_losses_train_gcamp,
	 boms_losses_test_gcamp, boms_man_losses_test_gcamp, boms_res_losses_test_gcamp, boms_mpe_losses_test_gcamp,
	 ) = train_models(bom,
					  filenames_gcamp,
					  discrete_states_gcamp,
					  eigenworms_gcamp,
					  traces_gcamp,
					  hyperparams=bom_reg_lambdas,
					  save_name=os.path.join('worm-2025-fieseler', 'new', ll, f'ss_bom_gcamp_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}'))

### HGM

In [ ]:
if gcamp:
	(hgms_params_gcamp,
	 hgms_losses_train_gcamp, hgms_man_losses_train_gcamp, hgms_res_losses_train_gcamp, hgms_mpe_losses_train_gcamp,
	 hgms_losses_test_gcamp, hgms_man_losses_test_gcamp, hgms_res_losses_test_gcamp, hgms_mpe_losses_test_gcamp,
	 ) = train_models(hgm,
					  filenames_gcamp,
					  discrete_states_gcamp,
					  eigenworms_gcamp,
					  traces_gcamp,
					  hyperparams=hgm_reg_lambdas,
					  save_name=os.path.join('worm-2025-fieseler', 'new', ll, f'ss_hgm_gcamp_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}'))

### losses

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_losses_train_gcamp, boms_losses_train_gcamp,
								  title='gcamp hgm vs bom: train',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_man_losses_train_gcamp, boms_man_losses_train_gcamp,
								  title='gcamp hgm vs bom: man train',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_res_losses_train_gcamp, boms_res_losses_train_gcamp,
								  title='gcamp hgm vs bom: res train',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_mpe_losses_train_gcamp, boms_mpe_losses_train_gcamp,
								  title='gcamp hgm vs bom: mpe train',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_losses_test_gcamp, boms_losses_test_gcamp,
								  title='gcamp hgm vs bom: test',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_man_losses_test_gcamp, boms_man_losses_test_gcamp,
								  title='gcamp hgm vs bom: man test',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_res_losses_test_gcamp, boms_res_losses_test_gcamp,
								  title='gcamp hgm vs bom: res test',
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits(filenames_gcamp,
								  hgms_mpe_losses_test_gcamp, boms_mpe_losses_test_gcamp,
								  title='gcamp hgm vs bom: mpe test',
								  ylimits=None)

# GFP

In [ ]:
if gfp:
	plot_utils.plot_traces(traces_gfp[plot_idx_gfp][selection_n_gfp],
						   discrete_states_gfp[plot_idx_gfp][['REV']],
						   label='reversal',
						   title=f'[{filenames_gfp[plot_idx_gfp]}] gfp traces', ylimits=None)

### BOM

In [ ]:
if gfp:
	(boms_params_gfp,
	 boms_losses_train_gfp, boms_man_losses_train_gfp, boms_res_losses_train_gfp, boms_mpe_losses_train_gfp,
	 boms_losses_test_gfp, boms_man_losses_test_gfp, boms_res_losses_test_gfp, boms_mpe_losses_test_gfp,
	 ) = train_models(bom,
					  filenames_gfp,
					  discrete_states_gfp,
					  eigenworms_gfp,
					  traces_gfp,
					  hyperparams=bom_reg_lambdas,
					  save_name=os.path.join('worm-2025-fieseler', 'new', ll, f'ss_bom_gfp_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}'))

### HGM

In [ ]:
if gfp:
	(hgms_params_gfp,
	 hgms_losses_train_gfp, hgms_man_losses_train_gfp, hgms_res_losses_train_gfp, hgms_mpe_losses_train_gfp,
	 hgms_losses_test_gfp, hgms_man_losses_test_gfp, hgms_res_losses_test_gfp, hgms_mpe_losses_test_gfp,
	 ) = train_models(hgm,
					  filenames_gfp,
					  discrete_states_gfp,
					  eigenworms_gfp,
					  traces_gfp,
					  hyperparams=hgm_reg_lambdas,
					  save_name=os.path.join('worm-2025-fieseler', 'new', ll, f'ss_hgm_gfp_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}'))

### losses

In [ ]:
if gfp:
	plot_utils.plot_losses_splits(filenames_gfp,
								  hgms_losses_train_gfp, boms_losses_train_gfp,
								  title='gfp hgm vs bom: train',
								  ylimits=None)

In [ ]:
if gfp:
	plot_utils.plot_losses_splits(filenames_gfp,
								  hgms_losses_test_gfp, boms_losses_test_gfp,
								  title='gfp hgm vs bom: test',
								  ylimits=None)

In [ ]:
if gfp:
	plot_utils.plot_losses_splits(filenames_gfp,
								  hgms_mpe_losses_train_gfp, boms_mpe_losses_train_gfp,
								  title='gfp hgm vs bom: mpe train',
								  ylimits=None)

In [ ]:
if gfp:
	plot_utils.plot_losses_splits(filenames_gfp,
								  hgms_mpe_losses_test_gfp, boms_mpe_losses_test_gfp,
								  title='gfp hgm vs bom: mpe test',
								  ylimits=None)